# Example 4: OpenFOAM velocity field in a 3D box

## Overview

This example replaces the Stokes velocity field from Example 3 with one computed by OpenFOAM. The geometry is a similar box with inlet and outlet channels, but extruded to 3D. The workflow is:

1. Extract the OpenFOAM case from the zip archive.
2. Read the mesh, boundary patches, and velocity field using [foam2dolfinx](https://github.com/festim-dev/foam2dolfinx).
3. Write everything to an io4dolfinx checkpoint.
4. Load the checkpoint and solve the advection-diffusion equation.

In [ ]:
import zipfile
from pathlib import Path

from mpi4py import MPI
from petsc4py import PETSc

import numpy as np

import ufl
import basix.ufl
import dolfinx
from dolfinx import fem
from dolfinx.fem.petsc import NonlinearProblem
import io4dolfinx
from foam2dolfinx import OpenFOAMReader

COMM = MPI.COMM_WORLD

## Extracting the foam data

The OpenFOAM case is stored in `foam_data.zip`. We extract it once; if `foam_data/` already exists the cell is a no-op.

In [12]:
if not Path("foam_data").exists():
    with zipfile.ZipFile("foam_data.zip", "r") as z:
        z.extractall(".")
    print("Extracted foam_data.zip")
else:
    print("foam_data/ already exists, skipping extraction")

foam_data/ already exists, skipping extraction


In [ ]:
foam_file = Path("foam_data/box.foam")
time_value = 11.1

reader = OpenFOAMReader(filename=foam_file, cell_type=10)
w_foam = reader.create_dolfinx_function_with_point_data(t=time_value, name="U")
msh = reader.dolfinx_meshes_dict["default"]
facet_mt = reader.create_facet_meshtags()

INLET_ID = 1
OUTLET_ID = 2
WALLS_ID = 3

## Saving to checkpoint

We write the mesh, facet tags, and velocity field to an [io4dolfinx](https://scientificcomputing.github.io/io4dolfinx/README.html) checkpoint so the advection-diffusion solve can be re-run without re-reading the OpenFOAM data.

In [15]:
CHECKPOINT = Path("foam_data/sim_checkpoint.bp")

io4dolfinx.write_mesh(CHECKPOINT, msh)
io4dolfinx.write_function(CHECKPOINT, w_foam, time=0.0, name="U")
io4dolfinx.write_meshtags(CHECKPOINT, msh, facet_mt, meshtag_name="facet_tags")

## Loading the velocity field

We reload the mesh, facet tags, and velocity from the checkpoint. The velocity was stored at degree 1 (CG1), matching the OpenFOAM node-centred format.

In [16]:
msh = io4dolfinx.read_mesh(CHECKPOINT, COMM)
msh.topology.create_connectivity(msh.topology.dim, msh.topology.dim - 1)
msh.topology.create_connectivity(msh.topology.dim - 1, msh.topology.dim)
facet_mt = io4dolfinx.read_meshtags(CHECKPOINT, msh, meshtag_name="facet_tags")

gdim = msh.geometry.dim
el = basix.ufl.element("Lagrange", msh.topology.cell_name(), 1, shape=(gdim,))
w = fem.Function(fem.functionspace(msh, el), name="U")
io4dolfinx.read_function(CHECKPOINT, w, time=0.0, name="U")

## Setting up the advection-diffusion problem

The formulation follows Example 3. The concentration $u$ is represented as a `fem.Function` (rather than a `TrialFunction`) so that the same nonlinear solver path used in Examples 1--3 is retained. The outlet uses a do-nothing condition, applying $\mathbf{w} \cdot \mathbf{n}\, u$ without upwind filtering, which is more robust when the flow has slight recirculation near the outlet step.

In [ ]:
V = fem.functionspace(msh, ("DG", 1))
u = fem.Function(V)
v_u = ufl.TestFunction(V)

n = ufl.FacetNormal(msh)
h = ufl.CellDiameter(msh)
ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_mt)
dS, dx = ufl.dS, ufl.dx

D = fem.Constant(msh, PETSc.ScalarType(1e-3))
penalty = fem.Constant(msh, PETSc.ScalarType(200))
u_inlet = fem.Constant(msh, PETSc.ScalarType(0.0))
f_source = fem.Constant(msh, PETSc.ScalarType(1.0))
lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

In [ ]:
F_outlet = ufl.inner(ufl.dot(w, n) * u, v_u) * ds(OUTLET_ID)

F_inlet = (
    D * (
        -ufl.inner(ufl.grad(u), v_u * n)
        - ufl.inner(ufl.grad(v_u), u * n)
        + (penalty / h) * ufl.inner(u, v_u)
    )
    - ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_inlet, v_u)
    + D * (
        ufl.inner(ufl.grad(v_u), u_inlet * n)
        - (penalty / h) * ufl.inner(u_inlet, v_u)
    )
) * ds(INLET_ID)

F = (
    -ufl.inner(w * u, ufl.grad(v_u)) * dx
    + ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v_u, n)) * dS
    + F_outlet
    + D * ufl.inner(ufl.grad(u), ufl.grad(v_u)) * dx
    - D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v_u, n)) * dS
    - D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v_u))) * dS
    + D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v_u, n)) * dS
    + F_inlet
    - ufl.inner(f_source, v_u) * dx
)

## Solving

In [ ]:
problem = NonlinearProblem(
    F,
    u,
    J=ufl.derivative(F, u),
    petsc_options_prefix="adv_diff",
    petsc_options={
        "snes_atol": 1e-12,
        "snes_rtol": 1e-12,
        "snes_max_it": 30,
        "ksp_type": "preonly",
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps",
    },
)
u = problem.solve()
u.x.scatter_forward()

from dolfinx.io import VTXWriter

writer = VTXWriter(COMM, "solution.bp", u, "BP5")
writer.write(t=0)

## Flux balance verification

In [ ]:
R = F
R_outlet = F_outlet
R_inlet = F_inlet

tdim = msh.topology.dim
fdim = tdim - 1
msh.topology.create_connectivity(fdim, tdim)
msh.topology.create_connectivity(tdim, tdim)
owned_size = V.dofmap.index_map.size_local * V.dofmap.index_map_bs


def get_owned_dofs(marker):
    facets = facet_mt.find(marker)
    if len(facets) == 0:
        return np.array([], dtype=np.int32)
    f_to_c = msh.topology.connectivity(fdim, tdim)
    cells = np.unique(np.concatenate([f_to_c.links(f) for f in facets]))
    dofs = fem.locate_dofs_topological(V, tdim, cells)
    return dofs[dofs < owned_size]


def compute_consistent_flux(residual_form, dofs):
    residual = fem.assemble_vector(fem.form(residual_form))
    residual.scatter_reverse(dolfinx.la.InsertMode.add)
    residual.scatter_forward()
    return msh.comm.allreduce(np.sum(residual.array[dofs]), op=MPI.SUM)


flux_outlet = -compute_consistent_flux(R - R_outlet, get_owned_dofs(OUTLET_ID))
flux_inlet = -compute_consistent_flux(R - R_inlet, get_owned_dofs(INLET_ID))
flux_wall = compute_consistent_flux(R, get_owned_dofs(WALLS_ID))

source_total = COMM.allreduce(
    fem.assemble_scalar(fem.form(f_source * dx)), op=MPI.SUM
)
consist_total = flux_outlet + flux_wall + flux_inlet


def pct(val):
    return 100 * val / source_total


print(f"Source integral : {source_total:.6e}")
print()
print(f"{'Flux outlet':20s} {flux_outlet:14.6e}")
print(f"{'Flux inlet':20s} {flux_inlet:14.6e}")
print(f"{'Flux wall':20s} {flux_wall:14.6e}")
print()
c_bal = source_total - consist_total
print(f"{'Total flux':20s} {consist_total:14.6e}")
print(f"{'Balance residual':20s} {c_bal:14.6e} ({pct(c_bal):+.2f}%)")